# Institution Resolver v3 — Colab: hibrit `decide` batch

Bu defter **tek bir isi** yapar: `benchmark_500_sample.csv` uzerinde hibrit karar
batch'ini kosar (once LLM'siz gate, gate emin degilse LLM hakem) ve sonucu
Drive'a yazar.

**Neden Colab:** yerel makinede Ollama (`gemma4:e4b`, ~9.6 GB) + Elasticsearch +
e5 embedding modeli ayni anda RAM'e sigmiyor. 2026-08-06 olcumu: 17.2 GB'lik
makinede swap 10.8/12.3 GB doldu, LLM yolu 4.07 s'den **20.7 s**'ye cikti (5.1x
yavaslama) ve 500 sorguluk kosu 22 dakika yerine ~97 dakikaya uzadi. Colab'da
model GPU'da calisir, ES ve embedding ayri bellekte durur.

**Calisma zamani:** `Runtime > Change runtime type > GPU` (T4 yeter). CPU'da da
calisir ama hakem cok yavaslar.

**Kalicilik:** ham veri, islenmis veri, **embedding cache'i**, Ollama modeli ve
ciktilar Drive'da tutulur. Oturum koparsa hicbiri yeniden uretilmez.

| Drive klasoru | Ne | Ilk kurulum | Sonraki oturumlar |
|---|---|---|---|
| `data_raw/` | `institution_parent.csv`, `institution_subunit.csv` (~361 MB) | **elle yuklenir** | hazir |
| `data_processed/` | kanonik JSONL + `embeddings.npz` (~808 MB) | ~10 dk + ~40 dk encode | hazir, yeniden uretilmez |
| `data_eval/` | `benchmark_500_sample.csv` (girdi) | **elle yuklenir** | hazir |
| `ollama_models/` | `gemma4:e4b` (~9.6 GB) | ~10 dk indirme | hazir |
| `output/` | batch sonuc CSV'si | — | `--resume` ile devam |

> **Ilk kurulumda elle yuklenmesi gerekenler:** `data_raw/` icine iki ham CSV,
> `data_eval/` icine `benchmark_500_sample.csv`. Ikisi de `.gitignore`'da,
> repodan gelmezler.

## 0) Calisma zamani kontrolu

In [ ]:
!nvidia-smi || echo "GPU YOK - hakem CPU'da calisir, cok yavas olur (Runtime > Change runtime type > GPU)"
!free -g | head -2

## 1) Drive baglama + kalici klasorler

`embeddings.npz` **`data_processed/` icinde** durur (kod onu
`data/processed/embeddings.npz` olarak yazar) - yani islenmis veriyi Drive'a
baglamak embedding cache'ini de kalici yapar. 712 MB'lik bu dosya ~40 dakikalik
GPU encode isini bir daha yaptirmaz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT      = "/content/drive/MyDrive/institution_resolver_v3"
DRIVE_RAW       = f"{DRIVE_ROOT}/data_raw"        # ham CSV'ler (elle yuklenir)
DRIVE_PROCESSED = f"{DRIVE_ROOT}/data_processed"  # kanonik JSONL + embeddings.npz
DRIVE_EVAL      = f"{DRIVE_ROOT}/data_eval"       # batch girdi CSV'leri (elle yuklenir)
DRIVE_JOBS      = f"{DRIVE_ROOT}/jobs"            # API job dosyalari
DRIVE_OLLAMA    = f"{DRIVE_ROOT}/ollama_models"   # LLM agirliklari (~9.6 GB)
DRIVE_OUTPUT    = f"{DRIVE_ROOT}/output"          # batch ciktilari

for p in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_EVAL, DRIVE_JOBS, DRIVE_OLLAMA, DRIVE_OUTPUT):
    os.makedirs(p, exist_ok=True)

def _rapor(etiket, yol):
    try:
        dosyalar = sorted(os.listdir(yol))
    except FileNotFoundError:
        dosyalar = []
    print(f"{etiket:12} {len(dosyalar):2} dosya")
    for f in dosyalar[:6]:
        boyut = os.path.getsize(os.path.join(yol, f)) / 1e6
        print(f"               - {f}  ({boyut:,.1f} MB)")

_rapor("raw:", DRIVE_RAW)
_rapor("processed:", DRIVE_PROCESSED)
_rapor("eval:", DRIVE_EVAL)
_rapor("ollama:", DRIVE_OLLAMA)

> Yukarida `raw` bos gorunuyorsa `institution_parent.csv` ve
> `institution_subunit.csv` dosyalarini Drive'da `institution_resolver_v3/data_raw/`
> icine yukleyin. `eval` bos gorunuyorsa `benchmark_500_sample.csv`'yi
> `data_eval/` icine koyun. Ikisi de `.gitignore`'da oldugu icin repodan gelmez.

## 2) Repo

In [ ]:
REPO_DIR = "/content/institution_resolver_v3"
BRANCH   = "feat/gate-asama1"   # calisma dali; ana dala gectiyseniz degistirin

import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/mcangultekin/institution_resolver_v3.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
%cd {REPO_DIR}
!git log --oneline -3

## 3) Veri dizinlerini Drive'a bagla

Repo icindeki `data/raw`, `data/processed`, `data/eval`, `data/jobs` dizinleri
Drive'daki kalici klasorlere sembolik baglanir. Bu sayede `inres3` komutlari
yol bilmeden calisir ve uretilen her sey (ozellikle `embeddings.npz`) oturum
kapaninca kaybolmaz.

In [ ]:
import os, shutil

def _link(name: str, target: str) -> None:
    """data/<name> -> Drive hedefi. Dizin zaten doluysa icerigi Drive'a TASIR
    (uzerine yazmadan), sonra sembolik baglar."""
    os.makedirs(target, exist_ok=True)
    link = f"data/{name}"
    if os.path.islink(link):
        return
    if os.path.isdir(link):
        for f in os.listdir(link):
            src, dst = f"{link}/{f}", f"{target}/{f}"
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(link)
    os.symlink(target, link)

os.makedirs("data", exist_ok=True)
_link("raw", DRIVE_RAW)
_link("processed", DRIVE_PROCESSED)
_link("eval", DRIVE_EVAL)
_link("jobs", DRIVE_JOBS)

!ls -la data
print()
npz = "data/processed/embeddings.npz"
if os.path.exists(npz):
    print(f"embedding cache HAZIR: {os.path.getsize(npz)/1e6:,.0f} MB "
          f"-> yeniden encode edilmeyecek")
else:
    print("embedding cache YOK -> ilk indeksleme ~40 dk surecek (bir kez)")

## 4) Python bagimliliklari

Colab'in kendi CUDA'li `torch`'u kullanilir - `Dockerfile`'daki CPU-only pin
burada gecerli DEGIL.

In [ ]:
%%bash
cd /content/institution_resolver_v3
pip install -q -e ".[dev,embed,llm,api]" 2>&1 | tail -3

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 5) Elasticsearch — native kurulum

Colab'da Docker daemon calismaz, ES dogrudan kurulur. Asagidaki uc ayar
2026-08-04'te canli dogrulanmis Colab'a ozgu duzeltmelerdir; kaldirmayin.

In [ ]:
%%bash
set -e
ES_VERSION=8.14.0
if [ ! -d /content/es ]; then
  wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-x86_64.tar.gz -O /content/es.tar.gz
  mkdir -p /content/es
  tar -xzf /content/es.tar.gz -C /content/es --strip-components=1
fi

grep -q '^discovery.type' /content/es/config/elasticsearch.yml || cat >> /content/es/config/elasticsearch.yml <<EOF
discovery.type: single-node
xpack.security.enabled: false
xpack.security.http.ssl.enabled: false
EOF

# (1) ML eklentisi cgroup istatistigi okurken Colab'in jupyter-children cgroup
# yapisinda AccessControlException atip ES'i cokertiyor - ML kullanmiyoruz.
grep -q '^xpack.ml.enabled' /content/es/config/elasticsearch.yml || echo 'xpack.ml.enabled: false' >> /content/es/config/elasticsearch.yml

# (2) Ayni istisna core OsService/MonitorService'ten de geliyor (ML'den bagimsiz,
# HER baslatmada) - /sys ve /proc okumasina acik izin.
cat > /content/es/config/elasticsearch.policy <<'POLEOF'
grant {
  permission java.io.FilePermission "/sys/-", "read";
  permission java.io.FilePermission "/proc/-", "read";
};
POLEOF

sysctl -w vm.max_map_count=262144 || true
id -u esuser &>/dev/null || useradd -m esuser
chown -R esuser:esuser /content/es

pkill -f 'org.elasticsearch.bootstrap.Elasticsearch' 2>/dev/null || true
sleep 1
# (3) setsid + </dev/null + disown: nohup tek basina Jupyter'in %%bash hucresini
# arka plan surecine bagli tutup sonsuza kadar bekletebiliyor.
sudo -u esuser env ES_JAVA_OPTS="-Xms2g -Xmx2g -Djava.security.policy=/content/es/config/elasticsearch.policy" \
  setsid /content/es/bin/elasticsearch < /dev/null > /content/es/es.log 2>&1 &
disown
sleep 3
pgrep -af 'org.elasticsearch.bootstrap.Elasticsearch' || echo 'UYARI: ES sureci gorunmuyor, /content/es/es.log kontrol et'

In [ ]:
import time, requests

for i in range(90):
    try:
        r = requests.get("http://localhost:9200", timeout=2)
        if r.status_code == 200:
            print("ES hazir:", r.json()["version"]["number"])
            break
    except Exception:
        pass
    time.sleep(2)
else:
    !tail -40 /content/es/es.log
    raise RuntimeError("ES 180s icinde ayaga kalkmadi - log yukarida")

## 6) Ollama + hakem modeli

Model agirliklari `OLLAMA_MODELS` ile Drive'a yazilir; ilk indirme ~10 dk,
sonraki oturumlarda aninda hazir.

In [ ]:
%%bash
set -e
# Colab'in temiz VM'inde zstd yok; Ollama kurulum betigi onsuz
# 'ERROR: This version requires zstd' ile duruyor (canli dogrulandi 2026-08-04).
apt-get update -qq && apt-get install -y -qq zstd
command -v ollama &> /dev/null || curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time, requests

os.environ["OLLAMA_MODELS"] = DRIVE_OLLAMA
subprocess.Popen(
    ["nohup", "ollama", "serve"],
    stdout=open("/content/ollama.log", "a"),
    stderr=subprocess.STDOUT,
    env=os.environ,
)

for _ in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Ollama 60s icinde ayaga kalkmadi, /content/ollama.log'a bak")

print("Ollama hazir")

In [ ]:
# config/default.yaml'daki judge.model ile AYNI olmali (gemma4:e4b).
!ollama pull gemma4:e4b
!ollama list

## 7) Veri isleme + indeksleme

Ikisi de **kosullu**: islenmis JSONL varsa yeniden uretilmez, `embeddings.npz`
varsa vektorler yeniden hesaplanmaz (cache id listesi + metin sha256'si ile
dogrulanir - bayat cache kullanilmaz).

Ilk kurulumda toplam ~50 dk; sonraki oturumlarda ~5 dk (sadece ES'e yukleme).

In [ ]:
%%bash
cd /content/institution_resolver_v3
set -e
if [ ! -f data/processed/parent_canonical.jsonl ]; then
  echo ">>> Islenmis veri yok, ham CSV'den uretiliyor (~10 dk)..."
  inres3 build-data --raw-dir data/raw --out-dir data/processed
else
  echo ">>> Islenmis veri hazir, atlaniyor."
fi

inres3 setup-es
inres3 index --embeddings

In [ ]:
import requests
h = requests.get("http://localhost:9200/institutions_v1/_search",
                 json={"size": 0, "track_total_hits": True,
                       "query": {"exists": {"field": "embedding"}}}, timeout=60).json()
print("embeddingli kayit:", h["hits"]["total"]["value"], "(231.291 bekleniyor)")

## 8) Saglik kontrolu — tek sorgu

In [ ]:
%%bash
cd /content/institution_resolver_v3
inres3 decide "gazi universitesi istatistik bolumu"

## 9) Hibrit batch — asil is

`decide-batch`: her sorgu once LLM'siz gate'ten gecer; parent VEYA subunit
`auto_match` vermezse sorgunun tamami LLM hakeme devredilir.

- `--workers 1` (varsayilan): **sirali**. Bu kosu ayni zamanda bir SURE
  baseline'i olacaksa sirali kalmali - paralellikte sorgular birbirinin suresini
  kirletir. Sadece karar analizi istiyorsaniz `--workers 2-4` ile hizlandirin.
- `--resume`: oturum koparsa ayni hucreyi tekrar calistirmak yeter, tamamlanan
  satirlar atlanir. (CSV basligi degisirse acik hata verir, sessizce bozmaz.)
- Cikti **Drive'da** - Colab oturumu olse de kalir.

In [ ]:
INPUT_CSV = f"{DRIVE_EVAL}/benchmark_500_sample.csv"
OUT_CSV   = f"{DRIVE_OUTPUT}/decide_baseline_dalga1.csv"
WORKERS   = 1     # sure baseline'i icin 1; sadece karar analizi icin 2-4

import os
assert os.path.exists(INPUT_CSV), f"Girdi CSV yok: {INPUT_CSV} (Drive'a yukleyin)"
print("girdi :", INPUT_CSV)
print("cikti :", OUT_CSV)

In [ ]:
!cd /content/institution_resolver_v3 && inres3 decide-batch "{INPUT_CSV}" \
    --out "{OUT_CSV}" --workers {WORKERS} --resume

## 10) Sonuc ozeti

Bu hucre kosu bitmeden de calistirilabilir - CSV her satirdan sonra
`flush` edildigi icin kismi sonuc okunabilir.

In [ ]:
import csv, statistics, collections

rows = list(csv.DictReader(open(OUT_CSV, newline="", encoding="utf-8")))
ok = [r for r in rows if r["status"] == "ok"]
print(f"islenen: {len(rows)}   ok: {len(ok)}   hata: {len(rows)-len(ok)}\n")

g = collections.defaultdict(list)
for r in ok:
    g[r["decided_by"]].append(float(r["elapsed_s"]))
for k, v in sorted(g.items()):
    print(f"  {k:6} n={len(v):4}  medyan={statistics.median(v):6.2f}s  "
          f"toplam={sum(v)/60:6.1f} dk  ({100*len(v)/len(ok):.0f}%)")

print("\nhatalar:")
for msg, n in collections.Counter(r["error"][:70] for r in rows if r["status"] == "error").items():
    print(f"  {n:3}  {msg}")

print("\nparent verdict:", dict(collections.Counter(r["parent_verdict"] for r in ok)))
print("subunit verdict:", dict(collections.Counter(r["subunit_verdict"] for r in ok)))

## 11) Gate ↔ hakem uyumu (Dalga 2'nin asil sorusu)

`gate_parent_id` / `gate_subunit_id` kolonlari **C4 ile eklendi** (2026-08-06).
Bunlar olmadan "gate hangi kaydi onermisti, hakemin sectigiyle AYNI MIYDI?"
sorusu cevaplanamiyordu - ve o soru iki karari kilitliyor:

- Gate `ambiguous` dedigi sorgularda hakem tek adaya baglaniyorsa **ve gate'in
  onerisi de ayni ise**, `any_rival_blocks_auto` bayragi kapatilabilir (o
  sorgular LLM'e hic gitmez). Oneriler FARKLI ise kapatmak yanlis auto uretir.
- Gate `review` deyip hakemin `auto_match` buldugu sorgular (upstream is).

In [ ]:
import csv, collections

rows = [r for r in csv.DictReader(open(OUT_CSV, newline="", encoding="utf-8"))
        if r["status"] == "ok"]
llm = [r for r in rows if r["decided_by"] == "judge"]
print(f"LLM'e giden: {len(llm)}/{len(rows)}  ({100*len(llm)/max(len(rows),1):.0f}%)\n")

print("=== Neden LLM'e gitti ===")
c = collections.Counter()
for r in llm:
    p, s = r["gate_parent_verdict"], r["gate_subunit_verdict"]
    if p != "auto_match" and s not in ("", "auto_match"): c["ikisi de"] += 1
    elif p != "auto_match": c["parent blokladi"] += 1
    else: c["SADECE subunit blokladi"] += 1
for k, v in c.most_common():
    print(f"  {v:4}  {k}")

print("\n=== gate verdict -> hakem verdict (parent) ===")
m = collections.Counter((r["gate_parent_verdict"], r["parent_verdict"]) for r in llm)
for (a, b), n in m.most_common():
    print(f"  gate={a:11} -> hakem={b:11} n={n:4}  {'AYNI' if a == b else 'degisti'}")

print("\n=== C4: gate'in ONERDIGI id hakemin sectigiyle ayni mi? ===")
for kova in ("ambiguous", "review", "auto_match", "no_match"):
    alt = [r for r in llm if r["gate_parent_verdict"] == kova]
    if not alt:
        continue
    ayni = sum(1 for r in alt if r["gate_parent_id"] and r["gate_parent_id"] == r["parent_id"])
    farkli = sum(1 for r in alt if r["gate_parent_id"] and r["gate_parent_id"] != r["parent_id"])
    idsiz = sum(1 for r in alt if not r["gate_parent_id"])
    print(f"  gate={kova:11} n={len(alt):4}  ayni={ayni:4}  farkli={farkli:4}  "
          f"gate id vermemis={idsiz:4}")
print("\nYORUM: 'ambiguous' satirlarinda ayni oran YUKSEK ve farkli SIFIRA yakinsa,")
print("       any_rival_blocks_auto guvenle kapatilabilir; farkli varsa KAPATMA.")

## Oturum koptuysa / yeniden baslarken

1. **1–6** arasi hucreleri sirayla calistir (Drive, repo, bagimliliklar, ES, Ollama).
   Islenmis veri, embedding cache'i ve model Drive'da oldugu icin bunlar hizli gecer.
2. **7**'yi calistir - `build-data` atlanir, `embeddings.npz` cache'ten okunur,
   sadece ES'e yukleme yapilir (~5 dk).
3. **9**'daki batch hucresini tekrar calistir: `--resume` tamamlanan satirlari atlar.

**Bilinen tuzaklar**
- Colab calisma zamani degisince (`GPU -> CPU`) hakem cok yavaslar; `nvidia-smi` ile dogrula.
- `inres3: command not found` -> 4. hucre (pip install -e) calistirilmamis.
- ES ayaga kalkmiyorsa `!tail -60 /content/es/es.log`.
- Ollama modeli her oturum yeniden iniyorsa `OLLAMA_MODELS` Drive'i gostermiyordur (6. hucre).
- Embedding yeniden hesaplaniyorsa `data/processed` sembolik bagi kopmustur (3. hucre).